This will:

Load meta-llama/Llama-2-7b-hf in 4-bit

Use sft_alpaca.jsonl to train LoRA adapters

Save adapters in checkpoints/qlora_finetuned_model/

🧠 Purpose:
This notebook fine-tunes meta-llama/Llama-2-7b-hf using QLoRA (Quantized LoRA) adapters on instruction-tuned Q&A pairs from sft_alpaca.jsonl (converted from questions_and_answers.json). It represents Pipeline P1: SFT Only.



Step 2: Import Libraries
Loads transformers, peft, and datasets.

BitsAndBytesConfig is used for quantization (4-bit training).

In [1]:
# 📌 Step 2: Import Libraries
from datasets import load_dataset
from transformers import (
    LlamaTokenizer, LlamaForCausalLM, TrainingArguments, Trainer,
    BitsAndBytesConfig
)
from peft import get_peft_model, LoraConfig, TaskType
import torch
import os
import time

Step 3: Load Dataset
Loads .jsonl fine-tuning data formatted in the Alpaca instruction style:

Instruction

Input

Response

This is the output of your earlier prepare_supervised_dataset.ipynb script.

In [2]:
# 📌 Step 3: Load Dataset
dataset = load_dataset("json", data_files="../data/sft_alpaca.jsonl", split="train")

Step 4: Load Tokenizer & Model (4-bit)
Uses LLaMA tokenizer and sets padding.

Loads LLaMA 2 7B with 4-bit quantization via bnb_config.

In [3]:
# 📌 Step 4: Load Tokenizer and Model in 4-bit with Optimized Config
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = LlamaTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16  # ✅ Fastest training mode
)

model = LlamaForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Step 5: Format + Tokenize Dataset
Combines instruction, input, and output into a single prompt.

Truncates to 256 tokens for GPU efficiency.

Copies input tokens as labels for causal language modeling.

In [4]:
# 📌 Step 5: Tokenize Dataset with Shorter Max Length
def format_instruction(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    tokenized = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(format_instruction)

Map:   0%|          | 0/635 [00:00<?, ? examples/s]

Step 6: Apply QLoRA Configuration
Adds LoRA adapters targeting q_proj and v_proj.

Sets LoRA dropout and rank.

In [6]:
# 📌 Step 6: Apply QLoRA Configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

Step 7: TrainingArguments
Full training config for 2 epochs.

Uses FP16 and paged_adamw_32bit optimizer for memory efficiency.

In [7]:
# 📌 Step 7: Training Arguments (Full run)
training_args = TrainingArguments(
    output_dir="../checkpoints/qlora_finetuned_model/",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,              # ✅ Full 2-epoch training
    logging_steps=10,
    learning_rate=2e-4,
    save_total_limit=1,
    save_steps=50,
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none"
)

Step 8: Training + Timer
Runs the training with Hugging Face Trainer.

Prints total runtime in minutes.

In [8]:
# 📌 Step 8: Train with Timer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)

start = time.time()
trainer.train()
print("⏱️ Total Training time (minutes):", round((time.time() - start) / 60, 2))

C:\Users\berfi\AppData\Local\Temp\ipykernel_34316\2841073525.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


  0%|          | 0/158 [00:00<?, ?it/s]

c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\llama\modeling_llama.py:602: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{'loss': 15.9889, 'grad_norm': inf, 'learning_rate': 0.00019367088607594938, 'epoch': 0.13}
{'loss': 3.3413, 'grad_norm': 6.994014739990234, 'learning_rate': 0.00018101265822784813, 'epoch': 0.25}
{'loss': 0.8381, 'grad_norm': 2.2828783988952637, 'learning_rate': 0.00016835443037974685, 'epoch': 0.38}
{'loss': 0.6203, 'grad_norm': 1.9801329374313354, 'learning_rate': 0.0001556962025316456, 'epoch': 0.5}
{'loss': 0.5556, 'grad_norm': 1.5141026973724365, 'learning_rate': 0.00014303797468354432, 'epoch': 0.63}
{'loss': 0.4403, 'grad_norm': 1.1949816942214966, 'learning_rate': 0.00013037974683544306, 'epoch': 0.75}
{'loss': 0.4192, 'grad_norm': 1.9345176219940186, 'learning_rate': 0.00011772151898734178, 'epoch': 0.88}
{'loss': 0.4477, 'grad_norm': 0.9641925096511841, 'learning_rate': 0.00010506329113924052, 'epoch': 1.01}
{'loss': 0.4959, 'grad_norm': 0.8878714442253113, 'learning_rate': 9.240506329113925e-05, 'epoch': 1.13}
{'loss': 0.4437, 'grad_norm': 1.101648211479187, 'learning_rate'

Step 9: Save Adapter
Saves the QLoRA adapter and tokenizer to ../checkpoints/qlora_finetuned_model/.

In [9]:
# 📌 Step 9: Save Adapters
model.save_pretrained("../checkpoints/qlora_finetuned_model/")
tokenizer.save_pretrained("../checkpoints/qlora_finetuned_model/")

print("✅ QLoRA fine-tuning complete. Adapters saved.")

✅ QLoRA fine-tuning complete. Adapters saved.


✅ Suggestions for Improvement

Area	Suggestion
File name	qlora_finetune_sft.ipynb
Dataset path	Document where sft_alpaca.jsonl came from (link to preprocessing)
Prompt structure	Optional: Make prompt formatting reusable (function or template file)
Logging	Save logs to a .txt or .json in the same folder for reproducibility
✅ Pipeline Reference

Pipeline	Description
P1	✅ SFT Only (QLoRA)